In [1]:
import pandas as pd
import bioframe as bf

In [2]:
chrom_sizes_file = "/project/fudenber_735/genomes/mm10/mm10.chrom.sizes.reduced"
dot_file = "/project/fudenber_735/GEO/bonev_2017_GSE96107/distiller-0.3.1_mm10/results/coolers/features/mustache_HiC_ES.mm10.mapq_30.10000.tsv"
autosomes_only = True
seq_length = 1310720

In [3]:
if autosomes_only:
    chromID_to_drop = ["chrX", "chrY", "chrM"]

In [4]:
def filter_by_chromID(
    df, chrom_column="chrom", chrID_to_drop=["chrX", "chrY", "chrM"]
):
    """
    Filter a DataFrame based on chromosome IDs.

    This function takes a pandas DataFrame and a list of chromosome IDs
    to drop from the DataFrame. It filters out rows where the 'chrom' column
    matches any of the provided chromosome IDs.

    Parameters:
    - df (pandas.DataFrame): Input DataFrame containing a 'chrom' column
                            representing chromosome IDs.
    - chrID_to_drop (list): List of chromosome IDs to be filtered out from the DataFrame.

    Returns:
    pandas.DataFrame: A new DataFrame with rows removed where the 'chrom' column
                      matches any of the chromosome IDs in chrID_to_drop.
    """
    filtered_df = df[~df[chrom_column].isin(chrID_to_drop)]
    return filtered_df


def filter_by_chrmlen(df, chrom_sizes_file, buffer_bp=0):
    """
    Filters a DataFrame of dot intervals so that neither anchor exceeds
    chromosome size bounds (with optional buffer), using plain pandas.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain: 'chrom', 'BIN1_START', 'BIN1_END', 'BIN2_START', 'BIN2_END'
    chrom_sizes_file : str or pd.DataFrame
        Path to chromosome sizes file (2-column TSV: chrom, size) or DataFrame
    buffer_bp : int
        Buffer to exclude near chromosome ends (in bp)

    Returns
    -------
    df_filtered : pd.DataFrame
        DataFrame with invalid rows removed
    """
    import pandas as pd

    assert isinstance(buffer_bp, int)

    # Read chrom sizes
    if isinstance(chrom_sizes_file, str):
        chrom_sizes_df = pd.read_csv(chrom_sizes_file, sep="\t", header=None, names=["chrom", "size"])
    else:
        chrom_sizes_df = chrom_sizes_file.rename(columns={0: "chrom", 1: "size"})

    # Convert to dictionary: {chrom: size}
    chrom_size_dict = dict(zip(chrom_sizes_df["chrom"], chrom_sizes_df["size"]))

    # Filtering function
    def is_valid(row):
        chrom = row['chrom']
        if chrom not in chrom_size_dict:
            return False
        chrom_len = chrom_size_dict[chrom]
        return (
            buffer_bp <= row['BIN1_START'] < row['BIN1_END'] <= chrom_len - buffer_bp and
            buffer_bp <= row['BIN2_START'] < row['BIN2_END'] <= chrom_len - buffer_bp
        )

    # Apply filtering
    df_filtered = df[df.apply(is_valid, axis=1)].copy()
    return df_filtered

In [5]:
# load dots (detected from HiC, 10kb resolution)
dots = pd.read_csv(dot_file, sep="\t")

In [6]:
len(dots)

9674

In [7]:
# dots[dots['BIN1_CHR'] != dots['BIN2_CHROMOSOME']]
# (sanity check) no interchromosomal dots
dots = dots.drop(columns=['BIN2_CHROMOSOME'])

# Rename BIN1_CHR to chrom
dots = dots.rename(columns={'BIN1_CHR': 'chrom'})

In [8]:
if autosomes_only:
    dots = filter_by_chromID(dots, chrID_to_drop=chromID_to_drop)

In [9]:
dots = filter_by_chrmlen(
    dots,
    chrom_sizes_file,
    seq_length,
)

In [10]:
def filter_by_anchor_distance(df, max_dist=256*2048):
    """
    Filters a DataFrame of dots to remove entries where the distance between
    BIN1_START and BIN2_START exceeds the maximum prediction window.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain: 'BIN1_START', 'BIN2_START'
    max_dist : int
        Maximum allowed genomic distance between anchors (in bp)

    Returns
    -------
    df_filtered : pd.DataFrame
        Filtered DataFrame where abs(BIN1_START - BIN2_START) <= max_dist
    """
    df_filtered = df[abs(df['BIN1_START'] - df['BIN2_START']) <= max_dist].copy()
    return df_filtered

In [11]:
len(dots)

9251

In [12]:
dots_distance_filtered = filter_by_anchor_distance(dots, max_dist=384 * 2048) # 3/4th of the map

In [14]:
384 * 2048

786432

In [13]:
len(dots_distance_filtered)

6818

In [15]:
len(dots_distance_filtered) / len(dots)

0.7370014052534861

In [ ]:
def enforce_anchor_order(df):
    """
    Ensures that BIN1_START < BIN2_START for each row.
    If not, swaps BIN1 and BIN2 columns.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain: 'BIN1_START', 'BIN1_END', 'BIN2_START', 'BIN2_END'

    Returns
    -------
    df_ordered : pd.DataFrame
        DataFrame with anchors ordered so that BIN1_START <= BIN2_START
    """
    df = df.copy()
    swap_mask = df['BIN1_START'] > df['BIN2_START']

    # Swap BIN1 and BIN2 values where needed
    for col1, col2 in [('BIN1_START', 'BIN2_START'), ('BIN1_END', 'BIN2_END')]:
        df.loc[swap_mask, [col1, col2]] = df.loc[swap_mask, [col2, col1]].values

    return df

In [ ]:
dots_distance_filtered = enforce_anchor_order(dots_distance_filtered)

In [ ]:
dots_distance_filtered.reset_index(drop=True, inplace=True)

In [ ]:
# 2nd dot anchor is not moving -> always centered at 384th bin
# since dots were called at 10kb resolution - let's assume 10kb -> ~5bins
# then the 2nd anchor starts with 382nd bin 

bin_size = 2048
cropping_applied = 64

rel_2anchor_start = (382 + cropping_applied) * bin_size
rel_2anchor_end = rel_2anchor_start + bin_size * 5

In [ ]:
to_window_end = seq_length - rel_2anchor_start

In [ ]:
dots_distance_filtered["window_start"] = dots_distance_filtered["BIN2_START"] - rel_2anchor_start
dots_distance_filtered["window_end"] = dots_distance_filtered["BIN2_START"] + to_window_end

In [ ]:
# (dots_distance_filtered["window_end"] - dots_distance_filtered["window_start"] != 1310720).any()

In [ ]:
dots_distance_filtered["anchor2_center_bin"] = [384 for i in range(len(dots_distance_filtered))]

# now, based on the dot1 - dot2 distance, we'll figure out which bin is center of the other dot

In [ ]:
dots_distance_filtered["anchors_dist"] = (dots_distance_filtered["BIN2_START"] - dots_distance_filtered["BIN1_START"]) // bin_size

In [ ]:
dots_distance_filtered["anchor1_center_bin"] = dots_distance_filtered["anchor2_center_bin"] - dots_distance_filtered["anchors_dist"]

In [ ]:
# sanity check
# dots_distance_filtered[dots_distance_filtered["anchor1_center_bin"] < 0]

In [ ]:
dots_distance_filtered.to_csv('/scratch1/smaruj/natural_dots/filtered_dots.tsv', sep='\t', index=False)